In [1]:
# 1. Imports

from pathlib import Path
import time


# ============================================================
# EXTERNE BIBLIOTHEKEN - Datenverarbeitung & Analyse
# ============================================================
import numpy as np                           # Numerische Berechnungen
import pandas as pd                          # Datenanalyse und TimeSeriesManagement
import ccxt                                  # Kryptobörsen-API (Binance etc.)

from dataclasses import dataclass
import configparser
import hmac
import hashlib
import requests
import pyarrow

In [13]:
# 2. Configuration

# Change run date
RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"


PROJECT_ROOT = Path.cwd().parent.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"
PROCESSED_DIR = DATA_DIR / "processed"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily"  
ASSET_OHLCV_5M_DIR = FETCHED_DIR / "asset_ohlcv_5m"

SRC_PATH = PROJECT_ROOT / "Src"
SECRET_PATH = SRC_PATH / "secret.cfg"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

OHLCV_INTERVAL = "5m"
BINANCE_KLINE_LIMIT = 1000
OHLCV_START_DATE = RUN_DATE
OHLCV_END_DATE_EXCLUSIVE = "2026-03-14"

def controlpath():
    print(f"Projektverzeichnis:            {PROJECT_ROOT}")
    print(f"Datenverzeichnis:              {DATA_DIR}")
    print(f"Rohdatenverzeichnis:           {RAW_DIR}")
    print(f"Gefetchte Daten:               {FETCHED_DIR}")
    print(f"Verarbeitete Daten:            {PROCESSED_DIR}")
    print(f"Source-Code:                   {SRC_PATH}")
    print(f"Secret:                        {SECRET_PATH}")
    print()

    checks = {
        "Assets-Datei":              ASSETS_PATH,
        "Asset-Daily-Verzeichnis":   ASSET_DAILY_PATH,
        "Asset-OHLCV-5M-Verzeichnis": ASSET_OHLCV_5M_DIR,
    }

    for name, path in checks.items():
        exists = path.exists()
        symbol = "✅" if exists else "❌ FEHLT"
        print(f"  {symbol}  {name}: {path}")

        # Falls Ordner → Inhalt anzeigen
        if exists and path.is_dir():
            files = sorted(path.glob("*.parquet"))
            if files:
                for f in files:
                    print(f"       📄 {f.name}")
            else:
                print(f"       ⚠️  Ordner ist leer — keine .parquet Dateien!")


OHLCV_5M_COLUMNS = [
    "timestamp",
    "asset_id",
    "exchange",
    "market_symbol",
    "open",
    "high",
    "low",
    "close",
    "volume_base",
    "volume_quote",
    "trade_count",
]

controlpath()

Projektverzeichnis:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit
Datenverzeichnis:              c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data
Rohdatenverzeichnis:           c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw
Gefetchte Daten:               c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched
Verarbeitete Daten:            c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\processed
Source-Code:                   c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src
Secret:                        c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\Src\secret.cfg

  ✅  Assets-Datei: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\assets.parquet
  ✅  Asset-Daily-Verzeichnis: c:\Users\ilija\Documents\OST\Semester_6\Bachlor\bachelorarbeit\data\raw\fetched\asset_daily
       📄 asset_daily.parquet
  ✅  Asset-

In [16]:
# =========================
# Konfiguration
# =========================
BASE_URL               = "https://data-api.binance.vision"
INTERVAL               = "5m"
SLEEP_BETWEEN_REQUESTS = 0.15

# =========================
# Public REST
# =========================
SESSION = requests.Session()

def public_get(path: str, params: dict | None = None):
    """
    Öffentlicher GET-Request an Binance API.
    Kein API-Key nötig — /api/v3/klines ist ein Public Endpoint.
    """
    url = f"{BASE_URL}{path}"
    r = SESSION.get(url, params=params, timeout=20)
    if r.status_code >= 400:
        raise RuntimeError(f"Binance Fehler {r.status_code}: {r.text}")
    return r.json()


# =========================
# Parquet Speicherung
# =========================
def json_to_parquet(data: list[dict]) -> None:
    """
    Speichert OHLCV-Daten als Parquet — pro Monat eine Datei.
    - Datei existiert bereits → zusammenführen + deduplizieren
    - Datei existiert nicht   → neu erstellen
    Sortierung: market_symbol → timestamp
    """
    ASSET_OHLCV_5M_DIR.mkdir(parents=True, exist_ok=True)

    df = pd.DataFrame(data)
    df["timestamp"] = df["timestamp"].dt.tz_convert("UTC")
    df["_year"]     = df["timestamp"].dt.year
    df["_month"]    = df["timestamp"].dt.month

    for (year, month), month_df in df.groupby(["_year", "_month"]):
        month_df = month_df.drop(columns=["_year", "_month"])
        out      = ASSET_OHLCV_5M_DIR / f"ohlcv_{year:04d}-{month:02d}.parquet"

        if out.exists():
            df_all = (
                pd.concat([pd.read_parquet(out), month_df], ignore_index=True)
                .drop_duplicates(subset=["timestamp", "market_symbol"])
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Aktualisiert: {out.name} ({len(df_all)} Zeilen)")
        else:
            df_all = (
                month_df
                .sort_values(["market_symbol", "timestamp"])
                .reset_index(drop=True)
            )
            print(f"  [PARQUET] Erstellt: {out.name} ({len(df_all)} Zeilen)")

        df_all.to_parquet(out, index=False)


# =========================
# Ticker-Liste laden
# =========================
def load_ticker_list() -> pd.DataFrame:
    """
    Lädt die bereits von 01_assets.ipynb gefilterten und
    gespeicherten Assets.

    01_assets.ipynb hat bereits folgende Filter angewendet:
      ✅ Top 100 Market Cap
      ✅ 24h Volumen > $50 Mio
      ✅ Gelistet auf mind. 10 Major Exchanges
      ✅ Auf Binance handelbar (binance_volume_24h_usd > 0)

    → Kein erneutes Filtern nötig!

    Rückgabe: DataFrame mit asset_id, symbol, market_symbol
    """
    # ── assets.parquet laden → enthält symbol ──
    df_assets = pd.read_parquet(ASSETS_PATH)
    print(f"[TICKER] Assets geladen: {len(df_assets)} Einträge")

    # ── asset_daily laden → neuesten Tag nehmen ──
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"[TICKER] Gefundene Daily-Dateien: {[f.name for f in parquet_files]}")

    df_daily = pd.concat(
        [pd.read_parquet(f) for f in parquet_files],
        ignore_index=True
    )

    # Neuesten Tag nehmen
    latest_date = df_daily["date"].max()
    df_latest   = df_daily[df_daily["date"] == latest_date].copy()
    print(f"[TICKER] Neuester Tag: {latest_date} ({len(df_latest)} Assets)")

    # ── Join: asset_id → symbol ──
    df = df_latest.merge(
        df_assets[["asset_id", "symbol", "name"]],
        on="asset_id",
        how="left"
    )

    # ── market_symbol bauen: SYMBOL + USDT ──
    # z.B. "BTC" → "BTCUSDT"
    # Nur Assets mit Binance-Volumen → sind auf Binance handelbar
    df = df[df["binance_volume_24h_usd"].notna()]
    df = df[df["binance_volume_24h_usd"] > 0]
    df["market_symbol"] = df["symbol"].str.upper() + "USDT"

    result = df[[
        "asset_id", "symbol", "name", "market_symbol",
        "market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"
    ]].reset_index(drop=True)

    print(f"\n[TICKER] Ticker-Liste ({len(result)} Assets):")
    print(result[["symbol", "market_symbol", "market_cap_usd"]].to_string())

    return result


# =========================
# Hauptfunktion
# =========================
def fetch_ohlcv_for_month() -> None:
    """
    Lädt historische 5m OHLCV-Daten von Binance für alle
    Assets aus asset_daily.parquet und speichert pro Monat
    eine Parquet-Datei.

    Voraussetzung: 01_assets.ipynb wurde bereits ausgeführt.
    Filter wurden dort bereits angewendet — hier kein erneutes Filtern.
    """
    print("=" * 60)
    print("Starte OHLCV-Datenbeschaffung")
    print("=" * 60)

    # ── Schritt 1: Ticker aus gespeicherten Dateien laden ──
    df_tickers = load_ticker_list()
    ticker_map = df_tickers.set_index("asset_id")["market_symbol"].to_dict()
    # ticker_map = {1: "BTCUSDT", 2: "ETHUSDT", ...}

    # ── Schritt 2: Asset-Daily Dateien durchgehen ──
    parquet_files = sorted(ASSET_DAILY_PATH.glob("*.parquet"))
    print(f"\n[FETCH] Monatsdateien: {[f.name for f in parquet_files]}")

    for file in parquet_files:
        print(f"\n[FETCH] Verarbeite: {file.name}")
        df_month         = pd.read_parquet(file)
        df_month["date"] = pd.to_datetime(df_month["date"]).dt.date

        # Nur Assets behalten die in ticker_map sind
        df_month = df_month[df_month["asset_id"].isin(ticker_map)]
        print(f"[FETCH] {len(df_month)} Zeilen")

        # ── Schritt 3: Pro Tag iterieren ──
        for current_date, df_day in df_month.groupby("date"):
            asset_ids = df_day["asset_id"].unique().tolist()
            symbols   = [ticker_map[aid] for aid in asset_ids if aid in ticker_map]
            print(f"\n[FETCH] {current_date}: {len(symbols)} Ticker")

            all_rows = []

            for market_symbol in symbols:
                try:
                    raw = public_get("/api/v3/klines", params={
                        "symbol":    market_symbol,
                        "interval":  INTERVAL,
                        "startTime": int(pd.Timestamp(current_date, tz="UTC").timestamp() * 1000),
                        "endTime":   int((pd.Timestamp(current_date, tz="UTC") + pd.Timedelta(days=1)).timestamp() * 1000),
                        "limit":     1000,  # 5m = max 288 Kerzen pro Tag
                    })

                    for row in raw:
                        all_rows.append({
                            "timestamp":     pd.to_datetime(int(row[0]), unit="ms", utc=True),
                            "market_symbol": market_symbol,
                            "open":          float(row[1]),
                            "high":          float(row[2]),
                            "low":           float(row[3]),
                            "close":         float(row[4]),
                            "volume_base":   float(row[5]),
                            "volume_quote":  float(row[7]),
                            "trade_count":   int(row[8]),
                        })

                    print(f"  [OK] {market_symbol}: {len(raw)} Kerzen")
                    time.sleep(SLEEP_BETWEEN_REQUESTS)

                except Exception as e:
                    print(f"  [FEHLER] {market_symbol}: {e}")

            # ── Schritt 4: Speichern ──
            if all_rows:
                print(f"  [SAVE] {len(all_rows)} Zeilen → Parquet")
                json_to_parquet(all_rows)
            else:
                print(f"  [SKIP] Keine Daten für {current_date}")

    print("\n" + "=" * 60)
    print("OHLCV-Datenbeschaffung abgeschlossen")
    print("=" * 60)


# =========================
# Ausführen
# =========================
fetch_ohlcv_for_month()

Starte OHLCV-Datenbeschaffung
[TICKER] Assets geladen: 100 Einträge
[TICKER] Gefundene Daily-Dateien: ['asset_daily.parquet']
[TICKER] Neuester Tag: 2026-03-13 (100 Assets)

[TICKER] Ticker-Liste (60 Assets):
     symbol market_symbol  market_cap_usd
0       BTC       BTCUSDT   1489151736234
1       ETH       ETHUSDT    284910320289
2      USDT      USDTUSDT    184033487448
3       XRP       XRPUSDT     93848427958
4       BNB       BNBUSDT     92814845420
5      USDC      USDCUSDT     79307642306
6       SOL       SOLUSDT     55094474070
7       TRX       TRXUSDT     28020678539
8      DOGE      DOGEUSDT     15685601122
9       ADA       ADAUSDT     10626275608
10      BCH       BCHUSDT      9641230635
11     LINK      LINKUSDT      7083258709
12     USDE      USDEUSDT      5922034835
13      XLM       XLMUSDT      5777615067
14      ZEC       ZECUSDT      4673427377
15     USD1      USD1USDT      4545402134
16      LTC       LTCUSDT      4530602409
17     AVAX      AVAXUSDT      4490

In [18]:
# =========================
# OHLCV Validierung
# =========================

EXPECTED_CANDLES_PER_DAY = 288  # 5m Intervall → 24h / 5min = 288


def load_ohlcv_month(path: Path) -> pd.DataFrame:
    """Lädt eine OHLCV Parquet-Datei und bereitet Timestamp vor."""
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["date"]      = df["timestamp"].dt.date
    return df


def clean_ohlcv_parquet(path: Path) -> None:
    """
    Entfernt fehlerhafte Kerzen die nicht zum Haupttag gehören.
    Haupttag = Tag mit den meisten Kerzen.
    Überschreibt die Datei bereinigt.
    """
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["date"]      = df["timestamp"].dt.date

    main_date = df.groupby("date").size().idxmax()
    foreign   = df[df["date"] != main_date]

    if not foreign.empty:
        print(f"[CLEAN] {path.name}: Entferne {len(foreign)} fremde Kerzen vom {[str(d) for d in foreign['date'].unique()]}")
        df = df[df["date"] == main_date].drop(columns=["date"]).reset_index(drop=True)
        df.to_parquet(path, index=False)
        print(f"[CLEAN] Gespeichert: {path.name} ({len(df)} Zeilen, Haupttag: {main_date})")
    else:
        print(f"[CLEAN] {path.name}: Keine fremden Kerzen — OK")


def validate_ohlcv_summary(df: pd.DataFrame) -> None:
    """Überblick: Kerzen, Ticker, Zeitraum, Volumen."""
    print("=" * 60)
    print("OHLCV ÜBERSICHT")
    print("=" * 60)
    print(f"  Kerzen gesamt      : {len(df):>10,}")
    print(f"  Ticker             : {df['market_symbol'].nunique():>10,}")
    print(f"  Tage               : {df['date'].nunique():>10,}")
    print(f"  Von                : {df['timestamp'].min()}")
    print(f"  Bis                : {df['timestamp'].max()}")
    print(f"  Volumen (quote) $  : {df['volume_quote'].sum():>15,.2f}")
    print()


def validate_ohlcv_ticker(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pro Ticker: Kerzen, Tage, Zeitraum, Vollständigkeit.
    Vollständigkeit = Ø der täglichen Abdeckung (nicht Gesamt / alle Tage).
    Verhindert falsche 50% bei Teiltagen.
    """
    rows = []
    for ticker, group in df.groupby("market_symbol"):
        days         = group["date"].nunique()
        daily_counts = group.groupby("date").size()

        # Ø Tagesabdeckung — fair auch wenn ein Tag nur 1 Kerze hat
        avg_pct       = round((daily_counts / EXPECTED_CANDLES_PER_DAY * 100).mean(), 1)
        complete_days = int((daily_counts >= 274).sum())  # 274 = 95% von 288
        status        = "✅ OK" if avg_pct >= 90 else "⚠️  Teilweise" if avg_pct >= 60 else "❌ Lücken"

        rows.append({
            "ticker":            ticker,
            "kerzen":            len(group),
            "tage":              days,
            "vollst. Tage":      complete_days,
            "von":               group["timestamp"].min().strftime("%Y-%m-%d"),
            "bis":               group["timestamp"].max().strftime("%Y-%m-%d"),
            "Ø vollst. %":       avg_pct,
            "status":            status,
        })

    result = pd.DataFrame(rows).sort_values("kerzen", ascending=False).reset_index(drop=True)
    print("=" * 60)
    print("TICKER-ANALYSE")
    print("=" * 60)
    print(result.to_string(index=False))
    print()
    return result


def validate_ohlcv_completeness(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pro Tag + Ticker: tatsächliche vs. erwartete Kerzen.
    Zeigt nur unvollständige Tage (< 95%).
    """
    rows = []
    for (ticker, date), group in df.groupby(["market_symbol", "date"]):
        pct    = round(len(group) / EXPECTED_CANDLES_PER_DAY * 100, 1)
        status = "✅" if pct >= 95 else "⚠️ " if pct >= 70 else "❌"
        rows.append({
            "datum":     date,
            "ticker":    ticker,
            "kerzen":    len(group),
            "erwartet":  EXPECTED_CANDLES_PER_DAY,
            "vollst. %": pct,
            "status":    status,
        })

    result     = pd.DataFrame(rows).sort_values(["vollst. %", "datum"]).reset_index(drop=True)
    incomplete = result[result["vollst. %"] < 95]

    print("=" * 60)
    print("VOLLSTÄNDIGKEIT (nur Tage < 95%)")
    print("=" * 60)
    if incomplete.empty:
        print("  ✅ Alle Tage vollständig!")
    else:
        print(incomplete.to_string(index=False))
    print()
    return result


def validate_ohlcv_quality(df: pd.DataFrame) -> None:
    """
    Qualitätsprüfungen:
    - Duplikate
    - Negative / NULL Werte
    - OHLC Konsistenz (high >= low, close zwischen low und high)
    - Zeitlücken > 10min (nur innerhalb desselben Tages)
    """
    checks = []

    # Duplikate
    dups = df.duplicated(subset=["timestamp", "market_symbol"]).sum()
    checks.append(("Keine Duplikate (timestamp + ticker)", dups == 0, f"{dups} Duplikate"))

    # Negative Werte
    for col in ["open", "high", "low", "close", "volume_base", "volume_quote"]:
        neg = (df[col] < 0).sum()
        checks.append((f"Keine negativen {col}-Werte", neg == 0, f"{neg} Fehler"))

    # NULL Werte
    nulls = df[["open","high","low","close","volume_base","volume_quote"]].isnull().sum().sum()
    checks.append(("Keine NULL Werte", nulls == 0, f"{nulls} NULL"))

    # OHLC Konsistenz
    hl_err = (df["high"] < df["low"]).sum()
    checks.append(("High >= Low", hl_err == 0, f"{hl_err} Fehler"))

    cl_err = ((df["close"] < df["low"]) | (df["close"] > df["high"])).sum()
    checks.append(("Close zwischen Low und High", cl_err == 0, f"{cl_err} Fehler"))

    # Zeitlücken — nur innerhalb desselben Tages prüfen
    # (Tageswechsel würde fälschlicherweise als Lücke gezählt)
    gap_tickers = []
    for (ticker, date), group in df.groupby(["market_symbol", "date"]):
        ts   = group["timestamp"].sort_values()
        gaps = (ts.diff().dt.total_seconds().dropna() > 600).sum()
        if gaps > 0:
            gap_tickers.append(f"{ticker} {date}({gaps})")
    checks.append(("Keine Zeitlücken > 10min", len(gap_tickers) == 0,
                   ", ".join(gap_tickers) if gap_tickers else "OK"))

    print("=" * 60)
    print("QUALITÄTSPRÜFUNGEN")
    print("=" * 60)
    for label, ok, detail in checks:
        icon = "✅" if ok else "❌"
        print(f"  {icon}  {label:<45} {detail}")
    print()


def validate_ohlcv_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Preisstatistiken pro Ticker."""
    stats = df.groupby("market_symbol").agg(
        min_close   = ("close", "min"),
        max_close   = ("close", "max"),
        avg_close   = ("close", "mean"),
        neg_values  = ("close", lambda x: (x < 0).sum()),
        null_values = ("close", lambda x: x.isnull().sum()),
    ).round(4).reset_index()

    print("=" * 60)
    print("PREISSTATISTIKEN")
    print("=" * 60)
    print(stats.to_string(index=False))
    print()
    return stats


def run_full_validation(path: Path) -> None:
    """
    Bereinigt und validiert eine OHLCV Parquet-Datei.
    Schritt 1: Fremde Kerzen entfernen (clean)
    Schritt 2: Alle Validierungen ausführen
    """
    print(f"\n📂 Validiere: {path.name}\n")

    # Zuerst bereinigen — entfernt Kerzen vom falschen Tag
    clean_ohlcv_parquet(path)

    # Dann frisch laden und validieren
    df = load_ohlcv_month(path)
    validate_ohlcv_summary(df)
    validate_ohlcv_ticker(df)
    validate_ohlcv_completeness(df)
    validate_ohlcv_quality(df)
    validate_ohlcv_stats(df)
    print("✅ Validierung abgeschlossen.")


# =========================
# Ausführen
# =========================
for file in sorted(ASSET_OHLCV_5M_DIR.glob("ohlcv_*.parquet")):
    run_full_validation(file)


📂 Validiere: ohlcv_2026-03.parquet

[CLEAN] ohlcv_2026-03.parquet: Entferne 59 fremde Kerzen vom ['2026-03-14']
[CLEAN] Gespeichert: ohlcv_2026-03.parquet (16992 Zeilen, Haupttag: 2026-03-13)
OHLCV ÜBERSICHT
  Kerzen gesamt      :     16,992
  Ticker             :         59
  Tage               :          1
  Von                : 2026-03-13 00:00:00+00:00
  Bis                : 2026-03-13 23:55:00+00:00
  Volumen (quote) $  : 7,619,028,521.50

TICKER-ANALYSE
     ticker  kerzen  tage  vollst. Tage        von        bis  Ø vollst. % status
   AAVEUSDT     288     1             1 2026-03-13 2026-03-13        100.0   ✅ OK
    TONUSDT     288     1             1 2026-03-13 2026-03-13        100.0   ✅ OK
  PENGUUSDT     288     1             1 2026-03-13 2026-03-13        100.0   ✅ OK
   PEPEUSDT     288     1             1 2026-03-13 2026-03-13        100.0   ✅ OK
    POLUSDT     288     1             1 2026-03-13 2026-03-13        100.0   ✅ OK
   PUMPUSDT     288     1             1 202